In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)

# Config
num_records = 5000
devices = ['TV', 'Mobile', 'Laptop', 'Tablet', 'moblie', 'lptop', None]  # with typos & missing
content_types = ['Movie', 'Series', 'Documentary']
day_variations = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday',
                  'monday', 'tuesday', 'wednesday']

data = []

for _ in range(num_records):
    user_id = fake.uuid4()
    
    # Random watch date (some missing)
    watch_date = fake.date_between(start_date='-6M', end_date='today') if random.random() > 0.05 else None
    
    # Day of week (with case variation)
    day_of_week = random.choice(day_variations)
    
    # Start time (mix 12h and 24h formats)
    start_hour = random.randint(18, 23) if random.random() > 0.2 else random.randint(0, 5)
    start_minute = random.randint(0, 59)
    if random.random() > 0.5:
        start_time = f"{start_hour}:{start_minute:02d}"
    else:
        start_time = datetime.strptime(f"{start_hour}:{start_minute:02d}", "%H:%M").strftime("%I:%M %p")
    
    # End time
    duration_hours = random.randint(1, 4)
    end_time_dt = (datetime.strptime(f"{start_hour}:{start_minute:02d}", "%H:%M") + timedelta(hours=duration_hours))
    if random.random() > 0.5:
        end_time = end_time_dt.strftime("%H:%M")
    else:
        end_time = end_time_dt.strftime("%I:%M %p")
    
    # Device type (some typos/missing)
    device_type = random.choice(devices)
    
    # Content type
    content_type = random.choice(content_types)
    
    # Episodes watched (with outliers)
    episodes = random.randint(1, 6)
    if random.random() < 0.02:
        episodes = random.randint(10, 15)  # binge outlier
    
    # Sleep hours (some missing + unrealistic)
    sleep_hours = round(random.uniform(4, 8), 1)
    if episodes > 8:
        sleep_hours = round(random.uniform(2, 4), 1)  # heavy binge -> less sleep
    if random.random() < 0.05:
        sleep_hours = None
    
    # Binge flag
    binge_flag = 1 if episodes > 3 else 0
    
    data.append([
        user_id, watch_date, day_of_week, start_time, end_time, device_type,
        content_type, episodes, sleep_hours, binge_flag
    ])

# Create DataFrame
df = pd.DataFrame(data, columns=[
    'user_id', 'watch_date', 'day_of_week', 'start_time', 'end_time', 'device_type',
    'content_type', 'episodes_watched', 'sleep_hours', 'binge_flag'
])

# Save
df.to_csv("netflix_vs_sleep.csv", index=False)
print("✅ netflix_vs_sleep.csv created with messy, realistic data!")


✅ netflix_vs_sleep.csv created with messy, realistic data!


Cleaning

In [2]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("netflix_vs_sleep.csv")

# 1️⃣ Handle missing values
# Fill missing watch_date with "Unknown" placeholder or drop if you prefer
df['watch_date'] = df['watch_date'].fillna("Unknown")
df['device_type'] = df['device_type'].fillna("Unknown")
df['sleep_hours'] = df['sleep_hours'].fillna(df['sleep_hours'].median())  # median imputation

# 2️⃣ Standardize day_of_week capitalization
df['day_of_week'] = df['day_of_week'].str.capitalize()

# 3️⃣ Fix device_type typos
device_corrections = {
    'moblie': 'Mobile',
    'lptop': 'Laptop',
    'Unknown': 'Unknown'
}
df['device_type'] = df['device_type'].replace(device_corrections)

# 4️⃣ Convert start_time and end_time to 24-hour format
def convert_to_24h(time_str):
    try:
        return pd.to_datetime(time_str, format="%I:%M %p").strftime("%H:%M")
    except:
        try:
            return pd.to_datetime(time_str, format="%H:%M").strftime("%H:%M")
        except:
            return None

df['start_time'] = df['start_time'].apply(convert_to_24h)
df['end_time'] = df['end_time'].apply(convert_to_24h)

# 5️⃣ Handle unrealistic episode counts
df['episodes_watched'] = df['episodes_watched'].apply(lambda x: x if x <= 10 else 10)

# 6️⃣ Ensure sleep_hours are realistic (between 0 and 12)
df['sleep_hours'] = df['sleep_hours'].apply(lambda x: x if 0 <= x <= 12 else np.nan)
df['sleep_hours'] = df['sleep_hours'].fillna(df['sleep_hours'].median())

# 7️⃣ Save cleaned dataset
df.to_csv("netflix_vs_sleep_cleaned.csv", index=False)

print("✅ Cleaning complete. File saved as netflix_vs_sleep_cleaned.csv")


✅ Cleaning complete. File saved as netflix_vs_sleep_cleaned.csv
